In [1]:
import datetime as dt

import numpy as np
import pandas as pd

from QuantStudio.Tools.Visualization import qs_help

In [2]:
import logging
import warnings
warnings.filterwarnings('ignore')

from QuantStudio.Core import setDefaultLogLevel
setDefaultLogLevel(level=logging.WARNING)

# AKShareDB

In [3]:
# 创建因子库对象并 connect
from QSExt.Factor.AKShareDB import AKShareDB

FDB = AKShareDB().connect()
print(qs_help(FDB))

2026-05-14 21:08:51,343 | QS | WARNING : 找不到配置文件: C:\Users\hst\QuantStudioConfig\AKShareDBConfig.json


类型: AKShareDB
模块: QSExt.Factor.AKShareDB
QS 对象类型: 因子库
QS 对象名称: AKShareDB
QSID: 77494f8400d4fd3c01f59ea8ef630702397ed232d73b18024f85dbde0c177c95
参数集:
    * Name(名称): <class 'str'>, 默认值 'AKShareDB', 当前取值: 'AKShareDB'
说明文档:
    基于 akshare 的因子库
    Document: https://akshare.akfamily.xyz/index.html
    GitHub: https://github.com/akfamily/akshare
    库配置信息文件在 Resource 目录下的 AKShareDBInfo.xlsx, 记录了相关配置信息


In [5]:
# 获取因子库中的因子表列表
print(FDB.TableNames[:5])

['东方财富-个股-股票信息', '实时行情数据-东财', '实时行情数据-新浪', '历史行情数据-东财', '历史行情数据-新浪']


## 获取时点序列

### 获取交易日序列

In [5]:
# 获取交易日序列的方法说明
print(qs_help(FDB.getTradeDay))

类型: method (bound to AKShareDB)
模块: QSExt.Factor.AKShareDB
签名: AKShareDB.getTradeDay(start_date: Optional[datetime.datetime] = None, end_date: Optional[datetime.datetime] = None, **kwargs) -> List[datetime.datetime]
说明文档:
    给定交易所、起始日和结束日, 获取交易日序列
    
    Args:
        start_date: 起始日, None 表示从可取的最早日期开始
        end_date: 结束日, None 表示当前日期
    
    Returns:
        交易日序列


In [6]:
# 给定起止时点, 获取交易日序列
DTs = FDB.getTradeDay(start_date=dt.datetime(2022, 1, 1), end_date=dt.datetime(2022, 1, 20))
print(DTs[:5])

[datetime.datetime(2022, 1, 4, 0, 0), datetime.datetime(2022, 1, 5, 0, 0), datetime.datetime(2022, 1, 6, 0, 0), datetime.datetime(2022, 1, 7, 0, 0), datetime.datetime(2022, 1, 10, 0, 0)]


## 获取证券代码序列

### 股票证券代码

In [7]:
# 获取股票证券代码方法说明
print(qs_help(FDB.getStockID))

类型: method (bound to AKShareDB)
模块: QSExt.Factor.AKShareDB
签名: AKShareDB.getStockID(exchange: Tuple[str] = ('SSE', 'SZSE', 'BSE'), **kwargs) -> List[str]
说明文档:
    给定交易所和日期, 获取股票证券代码序列, 包括已退市的
    
    Args:
        exchange: 交易所列表(tuple), 默认 ("SSE", "SZSE", "BSE") 表示上交所、深交所、北交所
        
    Returns:
        股票证券代码序列


In [ ]:
# 获取全体A股，包括已经退市的
IDs = FDB.getStockID()
print(IDs[:5])

# 因子表

In [4]:
# 获取因子表对象
FT = FDB.getTable("历史行情数据-东财", args={"LookBack": 0})
print(qs_help(FT))

类型: _DTRangeTable
模块: QSExt.Factor.AKShareDB
QS 对象类型: 计算节点-因子表
QS 对象名称: 历史行情数据-东财
QSID: 1366d4c65ee17fc6c28baeb382373cbe2541ee1589b1497cea65e3e0f14403a7
参数集:
    * Name(名称): <class 'str'>, 默认值 'Node', 当前取值: '历史行情数据-东财'
    * TableType(因子表类型): typing.Literal['DTRangeTable'], 默认值 'DTRangeTable', 当前取值: 'DTRangeTable'
    * IDAdj(ID调整): typing.Literal['去后缀', '无', '前缀'], 默认值 '去后缀', 当前取值: '去后缀'
    * DTFmt(时点格式): <class 'str'>, 默认值 '', 当前取值: ''
    * LookBack(回溯天数): <class 'int'>, 默认值 0, 当前取值: 0
说明文档:
    AKShareDB 库中基于取时间区间数据 API 的因子表


In [5]:
# 因子列表
print(FT.FactorNames)

['开盘', '收盘', '最高', '最低', '成交量', '成交额', '振幅', '涨跌幅', '涨跌额', '换手率']


## 读取数据

In [6]:
# 因子表读取数据
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ", "000002.SZ"]

Data = FT.readData(factor_names=["开盘", "收盘"], ids=IDs, dts=DTs)
print("因子表数据")
print(Data)

因子表数据
<class 'QuantStudio.Core.QSObject.Panel'>
Dimensions: 2 (items) x 5 (major_axis) x 2 (minor_axis)
Items axis: 开盘 to 收盘
Major_axis axis: 2025-01-01 00:00:00 to 2025-01-05 00:00:00
Minor_axis axis: 000001.SZ to 000002.SZ


# 因子

In [7]:
# 获取因子对象
F = FT.getFactor("收盘")
print(qs_help(F))

类型: Factor
模块: QuantStudio.Factor.Factor
QS 对象类型: 计算节点-因子
QS 对象名称: 收盘
QSID: f4c0e74f1aaecead16b594d35254b4e4a8469e7dfb3f53827772ed529593bc85
参数集:
    * Name(名称): <class 'str'>, 默认值 'Factor', 当前取值: '收盘'
    * Meta(元信息): <class 'dict'>, 默认值 {}, 当前取值: {}
    * SectionIDs(截面ID): typing.Optional[typing.List[str]], 默认值 None, 当前取值: None
    * CalcDTRuler(计算时点标尺): typing.Optional[typing.List[datetime.datetime]], 默认值 None, 当前取值: None
说明文档:
    因子对象
    因子可看做 DataFrame(index=[时点], columns=[ID])
    时点数据类型是 datetime, ID 的数据类型是 str


## 读取数据

In [8]:
# 因子读取数据
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ", "000002.SZ"]

Data = F.readData(ids=IDs, dts=DTs)
print(Data)

            000001.SZ  000002.SZ
2025-01-01        NaN        NaN
2025-01-02        NaN        NaN
2025-01-03        NaN        NaN
2025-01-04        NaN        NaN
2025-01-05        NaN        NaN
